# Chapter 4: Production ReliabilityEstimated time: ~7 hours.Prerequisites: Chapter 1 (`agentlib.llm_client`, the real/mock toggle, reused for thischapter's real rate-limit section).

## Setup

In [1]:
import sys
from pathlib import Path

_repo_root = Path.cwd()
if not (_repo_root / "agentlib").is_dir():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

import random
import time

from agentlib.grading import check
from agentlib import llm_client

random.seed(42)
print(f"LLM_PROVIDER = {llm_client.LLM_PROVIDER!r}, HAS_KEY = {llm_client.HAS_KEY}")


LLM_PROVIDER = 'anthropic', HAS_KEY = False


## Section 1: Definitions### Cache invalidation and TTLA **cache** stores the result of an expensive operation so repeated requests skip thework. A **TTL (time-to-live)** is the cache's built-in staleness budget: how many secondsan entry is trusted before it is treated as expired, whether or not it is actually stillcorrect. **Invalidation** is the explicit act of dropping an entry the moment the underlyingdata changes, rather than waiting for TTL to catch up.Real-world examples:- **Redis** TTL in production APIs: every key carries an expiry timestamp; after that  timestamp the key vanishes and the next read falls through to the database.- **Cloudflare** edge cache with purge-on-deploy: CDN nodes serve cached pages until a  deploy webhook tells them to drop the stale version immediately.- **DNS TTL**: a DNS record cached with a 300-second TTL means resolvers trust it for  five minutes. A domain migration that lowers TTL to 60 seconds before the cutover is  a textbook cache-invalidation strategy.### Retry with exponential backoff and jitter**Exponential backoff** spaces retries further and further apart (1s, 2s, 4s, 8s, ...)so a burst of failures does not hammer the dependency. **Jitter** adds randomness toeach delay so many clients retrying at once do not all land on the same schedule(the "thundering herd" problem).Real-world examples:- **AWS SDK** default retry behavior: exponential backoff with full jitter, capped at  20 seconds per attempt.- **Stripe API** retry recommendations: retry 429 and 5xx with exponential backoff;  do not retry 4xx (except 429).- **Google Cloud client libraries**: built-in retry with configurable backoff multiplier  and maximum delay.### Circuit breakerA **circuit breaker** (Nygard, "Release It!", 2007) prevents a caller from repeatedlyhitting a dependency that is fully down. Three states:- **Closed** (normal): calls pass through, consecutive failures are counted.- **Open** (tripped): calls fail immediately without reaching the dependency.- **Half-open** (probing): after a cooldown, one trial call is allowed through.  Success closes the breaker; failure reopens it.Real-world examples:- **Netflix Hystrix** (now Resilience4j): circuit breakers around every inter-service  call, with dashboards showing open/closed/half-open state per dependency.- **AWS Lambda** circuit breaker pattern: Lambda functions that stop calling a downstream  service after repeated timeouts, falling back to a cached or default response.- **Polly (.NET)**: a resilience library where CircuitBreakerPolicy wraps HTTP clients  with configurable failure thresholds and cooldown durations.### Staleness and graceful degradation**Staleness** is the gap between what your cache says and what is actually true right now.**Graceful degradation** is what a system does when a dependency fails and it cannot fullyrecover: it falls back to something reduced but useful (a stale-but-labeled answer, asimpler response) rather than failing outright.| Term | One-sentence definition ||---|---|| TTL | How long a cached entry is trusted before expiry || Invalidation | Dropping a cache entry the moment its source changes || Backoff | Spacing retries further apart on each failure || Jitter | Random noise added to backoff so retries do not synchronize || Circuit breaker | A state machine that stops calling a dead dependency || Graceful degradation | Serving a reduced response instead of a hard failure |

## Section 2: Concept Explanation### What changes between demo and productionA demo runs once, with curated input, on your machine, with nobody else hitting it at thesame time. Production runs continuously, with input you do not control, under concurrentload, against dependencies that sometimes fail. Nothing about the model changes betweendemo and production; everything about the environment around it does.Three patterns this chapter builds hands-on:| Pattern | What it solves | Analogy ||---|---|---|| Cache with TTL + invalidation | Serving fresh data without re-fetching on every request | Working from a printout that gets thrown away when the source changes || Retry with backoff + jitter | Recovering from a transient failure (flaky network, momentary overload) | Trying again after a hiccup, waiting longer each time instead of hammering immediately || Circuit breaker | Stopping calls to a dependency that is fully down | Noticing a vendor has not answered in three tries and stopping for a while |### Circuit breaker state machine```    calls succeed           failure count  +-------------+       reaches threshold       +-----------+  |             |  --------------------------->  |           |  |   CLOSED   |                                |   OPEN    |  |  (normal)  |  <---------------------------  | (fail     |  |             |     trial call succeeds        |  fast)    |  +-------------+                                +-----------+                                                    |    ^                              cooldown expires       |    |  trial call fails                              (one trial allowed)    v    |                                                +-----------+                                                | HALF-OPEN |                                                | (probing) |                                                +-----------+```### Exponential backoff timeline```  attempt 1 fails  |--- 1.0s + jitter ---|                         attempt 2 fails                         |------ 2.0s + jitter ------|                                                      attempt 3 fails                                                      |------------ 4.0s + jitter ------------|                                                                                                attempt 4 fails                                                                                                |--- 8.0s + jitter ---|```Each delay doubles. Linear growth (1s, 2s, 3s, 4s) looks almost identical over threeattempts and then stops helping: if a dependency is overloaded, backing off by a constantincrement never outpaces the queue building up in front of it.### Why 4xx is (mostly) not retryableHTTP 4xx means the client did something wrong. Retrying a 401 (bad credentials) or 404(resource not found) six times adds 30 seconds of latency before the caller sees theexact same error. Two 4xx codes are the exception:- **429** (Too Many Requests): the request is valid, the server is just rate-limiting.  Retry after backing off.- **408** (Request Timeout): the server gave up waiting. The request might succeed on  a faster retry.### Trade-offs| Decision | Short value | Long value ||---|---|---|| Cache TTL | Fresh but slow (more fetches) | Fast but potentially stale || Backoff base | Recovers faster from brief hiccups | Protects overloaded dependencies better || Circuit breaker threshold | Trips early (fewer wasted calls) | Tolerates brief hiccups without tripping || Circuit breaker cooldown | Recovers faster after outage | Avoids re-tripping on a slow recovery |

## Section 3: Example Code SegmentsA fake mini-codebase with version history, a flaky-tool generator, and adead-dependency simulator. These are the building blocks for the graded tasksand break-it scenarios that follow.

### A fake mini-codebase with version historyFour files with realistic docstrings, plus a version history per file: a list of`(timestamp, content)` pairs simulating real edits over time. This is what an AIcoding assistant would be indexing in production, and it is the thing that goes stale.

In [2]:
CODEBASE_HISTORY = {
    "billing.py": [
        (0, '''def calculate_total(items):
    """Sum the price of every item in the cart. No discount support."""
    return sum(item["price"] for item in items)
'''),
        (3600, '''def calculate_total(items, discount_code=None):
    """Sum the price of every item in the cart, applying a discount code if provided."""
    total = sum(item["price"] for item in items)
    if discount_code == "SAVE10":
        total *= 0.9
    return total
'''),
    ],
    "auth.py": [
        (0, '''def authenticate(username, password):
    """Check a username/password pair against the user table. No rate limiting."""
    return _lookup_user(username) and _check_password(username, password)
'''),
    ],
    "notifications.py": [
        (0, '''def send_email(to, subject, body):
    """Send a transactional email via the configured provider."""
    return _provider.send(to=to, subject=subject, body=body)
'''),
    ],
    "inventory.py": [
        (0, '''def reserve_stock(sku, quantity):
    """Reserve stock for an order. Raises InsufficientStockError if unavailable."""
    if _available(sku) < quantity:
        raise InsufficientStockError(sku)
    _decrement(sku, quantity)
'''),
    ],
}


class FileStore:
    """Simulates a version-controlled codebase: get_file(name, at_time) returns whatever
    content was current at that timestamp."""

    def __init__(self, history: dict):
        self.history = history

    def get_file(self, filename: str, at_time: float) -> str:
        current = None
        for ts, content in self.history[filename]:
            if ts <= at_time:
                current = content
        return current


filestore = FileStore(CODEBASE_HISTORY)
print(filestore.get_file("billing.py", at_time=0))


def calculate_total(items):
    """Sum the price of every item in the cart. No discount support."""
    return sum(item["price"] for item in items)



### A flaky tool (configurable failure rate)Wraps a function that fails with a given probability. Deterministic via seed.

In [7]:
def make_flaky_tool(fail_probability: float, seed: int):
    rng = random.Random(seed)

    def flaky_tool():
        if rng.random() < fail_probability:
            raise ConnectionError("simulated transient failure")
        return "success"

    return flaky_tool


print("--- Bug: no retry, one failure ends the task ---\n")
flaky = make_flaky_tool(fail_probability=0.7, seed=1)
try:
    result = flaky()
    print("Result:", result)
except Exception as exc:
    print(f"Failed on the first attempt: {exc}")
    print("A 70%-fail-rate dependency will kill most single-shot calls to it -- unacceptable")
    print("for anything an agent needs to complete reliably.")


--- Bug: no retry, one failure ends the task ---

Failed on the first attempt: simulated transient failure
A 70%-fail-rate dependency will kill most single-shot calls to it -- unacceptable
for anything an agent needs to complete reliably.


### A dead dependency (always fails)For testing circuit breaker behavior: a dependency that never succeeds.

In [12]:
def make_always_failing_tool():
    def broken_tool():
        raise ConnectionError("dependency is down")
    return broken_tool


print("--- Bug: naive caller keeps hammering a fully-dead dependency ---\n")
broken = make_always_failing_tool()
attempted = 0
for t in range(10):
    try:
        broken()
    except Exception:
        pass
    attempted += 1
print(f"Made {attempted} calls to a dependency that never once succeeded.")
print("Every one of those wastes time on both sides -- a real network call here would add")
print("real latency to each failure, and do it 10 times over for zero benefit.")


--- Bug: naive caller keeps hammering a fully-dead dependency ---

Made 10 calls to a dependency that never once succeeded.
Every one of those wastes time on both sides -- a real network call here would add
real latency to each failure, and do it 10 times over for zero benefit.


## Section 4: Build It YourselfFour graded tasks: a TTL cache, exponential backoff with jitter, a retry predicate thatdistinguishes transient from permanent errors, and a circuit breaker with three states.

### Task 1: `TTLCache` (cache with time-to-live)The cache is a dictionary with an expiry clock. The only decision that matters is whenthe clock gets consulted.Cache an answer at 9am with a two-hour TTL, then read it at 3pm. Nothing was written inbetween, so nothing prompted the cache to reconsider anything. If expiry is only evaluatedat write time, that 9am answer is still sitting there at 3pm, six hours stale. Stalenessis a property of the read.

In [ ]:
class TTLCache:
    '''A cache whose entries go stale on a clock.

    There is no wall clock here: every method takes `now` explicitly, which is what makes
    the staleness behaviour testable without sleeping through a real TTL.

    - get(key, now) -> (value, cached_at) if the entry exists and is younger than ttl
                       seconds, else (None, None). An entry that has reached exactly ttl
                       seconds is expired.
    - set(key, value, now) -> store the value, stamped with `now`. Rewriting a key restarts
                       its clock.
    - invalidate(key) -> drop the entry; a key that was never cached is not an error.
    '''

    def __init__(self, ttl_seconds: float):
        self.ttl = ttl_seconds
        self.store = {}  # key -> (value, cached_at)

    def get(self, key, now: float):
        raise NotImplementedError("Implement me, then re-run this cell")

    def set(self, key, value, now: float):
        raise NotImplementedError("Implement me, then re-run this cell")

    def invalidate(self, key):
        raise NotImplementedError("Implement me, then re-run this cell")


TTLCache = check("ch04-ttl-cache", TTLCache)

#### What the cache does once it passes

In [4]:
cache = TTLCache(ttl_seconds=7200)  # 2-hour TTL
value, cached_at = cache.get("billing.py", now=0)
print("Before anything is cached:", value, cached_at)
cache.set("billing.py", filestore.get_file("billing.py", at_time=0), now=0)
value, cached_at = cache.get("billing.py", now=10)
print("After caching, a quick re-check:", value[:40].strip(), "...", "cached_at =", cached_at)

Before anything is cached: None None
After caching, a quick re-check: def calculate_total(items):
    """Sum t ... cached_at = 0


### Task 2: `retry_with_backoff` (exponential backoff with jitter)The flaky tool below fails 70% of the time. A single-shot call almost always fails.Your wrapper retries with exponential delays.

In [ ]:
def retry_with_backoff(fn, max_retries: int = 6, base_delay: float = 1.0, jitter: float = 0.5,
                        seed: int = 1, sleep_fn=None, retry_predicate=None):
    '''Reusable exponential backoff + jitter wrapper.

    Returns a `wrapped(*args, **kwargs)` that forwards through to fn and returns
    (result, attempts_log). Each failed attempt appends
    {"attempt": n, "error": str(exc), "backoff_s": round(delay, 2)} to that log.

    The delay before retry number `attempt` (1-based) is:

        base_delay * 2 ** (attempt - 1) + rng.uniform(0, jitter)

    Exponential, not linear. Linear growth looks almost identical over three attempts and
    then stops helping precisely when it matters: if a dependency is overloaded, backing off
    by a constant increment never outpaces the queue building up in front of it.

    Back off AFTER a failure. A call that succeeds first time must not sleep at all.

    sleep_fn defaults to None, which logs the delay each attempt WOULD have taken without
    actually sleeping (keeps this notebook fast); pass sleep_fn=time.sleep for real behavior
    against a real dependency. Use random.Random(seed) so runs are reproducible.

    retry_predicate, if given, is called with the exception; a False verdict re-raises
    immediately instead of retrying (see the next cell). Exhausting max_retries raises
    RuntimeError `from` the last exception.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


retry_with_backoff = check("ch04-backoff", retry_with_backoff)

#### What backoff looks like in practice

In [9]:
print("--- Fix: retry with exponential backoff + jitter ---\n")
robust_flaky = retry_with_backoff(make_flaky_tool(fail_probability=0.7, seed=1), max_retries=8)
result, attempts_log = robust_flaky()
for a in attempts_log:
    print(f"  attempt {a['attempt']} failed ({a['error']}) -- backed off {a['backoff_s']}s before retrying")
print(f"\nResult: {result!r}, succeeded on attempt {len(attempts_log) + 1}")

--- Fix: retry with exponential backoff + jitter ---

  attempt 1 failed (simulated transient failure) -- backed off 1.07s before retrying

Result: 'success', succeeded on attempt 2


### Task 3: `should_retry` (retry predicate)Backoff answers how long to wait. This function answers whether to wait at all. Amalformed request (400) is exactly as malformed on the sixth attempt as the first;six exponential backoffs just add half a minute of latency before the caller sees theerror it could have had immediately.

In [ ]:
class ApiError(Exception):
    '''Stands in for a provider SDK's HTTP error, which carries a status code.'''

    def __init__(self, status_code, message="api error"):
        super().__init__(f"{status_code}: {message}")
        self.status_code = status_code


def should_retry(error: Exception) -> bool:
    '''Is retrying this error worth anything, or will it fail identically every time?

    Retry: transport failures with no status at all (ConnectionError, TimeoutError), the
    5xx range, and the two transient 4xx statuses, 408 and 429.

    Don't retry: the rest of the 4xx range -- 400, 401, 403, 404, 422 all describe a request
    that is broken in a way another attempt cannot fix.

    An error carrying no status_code at all is treated as transient.
    '''
    raise NotImplementedError("Implement me, then re-run this cell")


should_retry = check("ch04-no-retry-4xx", should_retry)

#### The predicate in action

In [11]:
def make_status_tool(status_code):
    def tool():
        raise ApiError(status_code)
    return tool


print("--- The same wrapper, now asking whether the error is worth retrying at all ---\n")
for status in (429, 400):
    attempted = {"n": 0}

    def counted():
        attempted["n"] += 1
        make_status_tool(status)()

    guarded = retry_with_backoff(counted, max_retries=5, retry_predicate=should_retry)
    try:
        guarded()
    except Exception as exc:
        print(f"  HTTP {status}: gave up after {attempted['n']} attempt(s) -- {type(exc).__name__}")

print()
print("A 429 is worth five attempts; a 400 is worth exactly one. Without the predicate both")
print("cost the same five attempts and the same total backoff, and the 400's caller waits")
print("through all of it to be told something the very first response already said.")

--- The same wrapper, now asking whether the error is worth retrying at all ---

  HTTP 429: gave up after 5 attempt(s) -- RuntimeError
  HTTP 400: gave up after 1 attempt(s) -- ApiError

A 429 is worth five attempts; a 400 is worth exactly one. Without the predicate both
cost the same five attempts and the same total backoff, and the 400's caller waits
through all of it to be told something the very first response already said.


### Task 4: `CircuitBreaker` (three-state failure isolation)The half-open transition is the part that is easy to leave out and impossible to noticeyou left out, because everything looks correct right up until the dependency recovers.

In [ ]:
class CircuitBreaker:
    '''States: closed (normal) -> open (failing fast, not calling the dependency at all)
    -> half-open (after a cooldown, allow one trial call through) -> closed on success,
    back to open on failure.

    call(fn, now) is the whole interface. While closed, it calls fn, counts CONSECUTIVE
    failures (a success resets the count), and opens once the count reaches
    failure_threshold, recording `now` as opened_at. While open, it raises immediately
    WITHOUT calling fn -- that is the entire saving.

    The half-open transition is the part that is easy to leave out and impossible to notice
    you left out, because everything looks correct right up until the dependency recovers.
    Once `cooldown` seconds have elapsed since opened_at, exactly one trial call must be let
    through. Succeed and the circuit closes; fail and it opens again with a fresh cooldown.
    Without it the breaker never re-tests anything, and the outage outlives its own cause.

    Failures reaching fn still re-raise, so the caller sees the real error.
    '''

    def __init__(self, failure_threshold: int = 3, cooldown: float = 5):
        self.failure_threshold = failure_threshold
        self.cooldown = cooldown
        self.failure_count = 0
        self.state = "closed"
        self.opened_at = None

    def call(self, fn, now: float):
        raise NotImplementedError("Implement me, then re-run this cell")


CircuitBreaker = check("ch04-circuit-breaker", CircuitBreaker)

#### Breaker in action: failing fast

In [14]:
breaker = CircuitBreaker(failure_threshold=3, cooldown=5)
underlying_calls = 0


def counted_broken():
    global underlying_calls
    underlying_calls += 1
    return make_always_failing_tool()()


print("--- Fix: circuit breaker fails fast instead of hammering a dead dependency ---\n")
for t in range(10):
    try:
        breaker.call(counted_broken, now=t)
    except Exception as exc:
        print(f"t={t}: {exc} (breaker state: {breaker.state})")

print(f"\nUnderlying dependency was actually invoked {underlying_calls} times out of 10 attempts")
print("(vs. 10/10 in the buggy version above) -- the breaker opened after 3 failures and")
print("stopped calling the dependency at all for the rest of the window.")

--- Fix: circuit breaker fails fast instead of hammering a dead dependency ---

t=0: dependency is down (breaker state: closed)
t=1: dependency is down (breaker state: closed)
t=2: dependency is down (breaker state: open)
t=3: circuit open -- failing fast (dependency not called) (breaker state: open)
t=4: circuit open -- failing fast (dependency not called) (breaker state: open)
t=5: circuit open -- failing fast (dependency not called) (breaker state: open)
t=6: circuit open -- failing fast (dependency not called) (breaker state: open)
t=7: dependency is down (breaker state: open)
t=8: circuit open -- failing fast (dependency not called) (breaker state: open)
t=9: circuit open -- failing fast (dependency not called) (breaker state: open)

Underlying dependency was actually invoked 4 times out of 10 attempts
(vs. 10/10 in the buggy version above) -- the breaker opened after 3 failures and
stopped calling the dependency at all for the rest of the window.


#### Recovery: the dependency comes back up

In [15]:
print("--- Recovery: the dependency comes back up, the breaker notices via one trial call ---\n")

recovery_calls = 0


def working_tool():
    global recovery_calls
    recovery_calls += 1
    return "success"


recovery_time = breaker.opened_at + breaker.cooldown + 1
result = breaker.call(working_tool, now=recovery_time)
print(f"t={recovery_time}: call succeeded ({result!r}), breaker state is now {breaker.state!r}")
print(f"Only {recovery_calls} trial call was needed to detect recovery -- not a burst of retries.")


--- Recovery: the dependency comes back up, the breaker notices via one trial call ---

t=13: call succeeded ('success'), breaker state is now 'closed'
Only 1 trial call was needed to detect recovery -- not a burst of retries.


## Section 5: PlaygroundExperiments with editable parameters. Change the values marked `EDIT THESE`, re-run thecell, and observe how the behavior changes.

### Experiment 1: TTL durationHow does TTL length affect cache freshness? Try different TTL values and observe whenthe cache serves stale data vs. fetching fresh data.

In [ ]:
# --- EDIT THESE ---TTL_VALUES = [1, 60, 3600, 7200]  # try [10, 300, 1800, 86400]print(f"{'TTL (s)':>10}  {'Stale at t=3700?':>18}  {'Reason'}")print("-" * 60)for ttl in TTL_VALUES:    test_cache = TTLCache(ttl_seconds=ttl)    test_cache.set("billing.py", filestore.get_file("billing.py", at_time=0), now=0)    val, ts = test_cache.get("billing.py", now=3700)    if val is None:        stale = "No (expired)"        reason = f"TTL={ttl}s < age=3700s, entry expired"    elif "discount_code" not in val:        stale = "YES (stale)"        reason = f"TTL={ttl}s >= age=3700s, serving pre-edit version"    else:        stale = "No (fresh)"        reason = "fetched after edit"    print(f"{ttl:>10}  {stale:>18}  {reason}")print()print("Short TTL: more cache misses (slower) but fresher data.")print("Long TTL: fewer misses (faster) but stale data survives longer.")

### Experiment 2: Backoff base and max retriesHow does the backoff base affect total wait time? The total worst-case wait beforegiving up is: sum of base * 2^i for i in 0..max_retries-1, plus jitter.

In [ ]:
# --- EDIT THESE ---BASE_DELAYS = [0.5, 1.0, 2.0, 5.0]  # try [0.1, 0.5, 1.0, 3.0]MAX_RETRIES_LIST = [3, 5, 8]  # try [2, 4, 6, 10]print(f"{'Base (s)':>10}  {'Max retries':>12}  {'Worst-case wait (s)':>20}")print("-" * 50)for base in BASE_DELAYS:    for max_r in MAX_RETRIES_LIST:        total = sum(base * 2**i for i in range(max_r))        print(f"{base:>10.1f}  {max_r:>12}  {total:>20.1f}")    print()print("A base of 1.0s with 8 retries means up to 255s worst-case wait.")print("A base of 0.5s with 3 retries means only 3.5s -- fast, but gives up early.")

### Experiment 3: Circuit breaker threshold sensitivityHow many failures before the breaker trips? A low threshold trips early (fewer wastedcalls but more false positives); a high threshold tolerates brief hiccups.

In [ ]:
# --- EDIT THESE ---THRESHOLDS = [1, 3, 5, 10]  # try [2, 4, 8]NUM_ATTEMPTS = 15print(f"{'Threshold':>10}  {'Calls to dep':>13}  {'Fast-fails':>11}")print("-" * 40)for thresh in THRESHOLDS:    b = CircuitBreaker(failure_threshold=thresh, cooldown=100)    dep_calls = 0    fast_fails = 0    for t in range(NUM_ATTEMPTS):        try:            def _counted():                nonlocal dep_calls                dep_calls += 1                raise ConnectionError("down")            b.call(_counted, now=t)        except RuntimeError:            fast_fails += 1        except ConnectionError:            pass    print(f"{thresh:>10}  {dep_calls:>13}  {fast_fails:>11}")print()print("Lower threshold = fewer wasted calls to a dead dependency.")print("Higher threshold = more tolerance for brief transient failures.")

### Experiment 4: Circuit breaker cooldown and recoveryHow long should the breaker stay open before probing? Short cooldown recovers fasterbut risks re-tripping if the dependency is still down.

In [ ]:
# --- EDIT THESE ---COOLDOWNS = [2, 5, 15, 60]  # try [1, 10, 30, 120]# Simulate: dependency is down for exactly 10 seconds, then recoversDOWN_DURATION = 10print(f"{'Cooldown':>10}  {'Recovery detected at t=':>24}")print("-" * 40)for cd in COOLDOWNS:    b = CircuitBreaker(failure_threshold=2, cooldown=cd)    # Trip the breaker at t=0, t=1    for t in range(2):        try:            b.call(lambda: (_ for _ in ()).throw(ConnectionError("down")), now=t)        except:            pass    # Probe at each cooldown interval until recovery    detected_at = None    for probe_t in range(2, 100):        if b.state == "open" and b.opened_at is not None and probe_t < b.opened_at + cd:            continue        try:            def _maybe_works(t=probe_t):                if t < DOWN_DURATION:                    raise ConnectionError("still down")                return "up"            b.call(_maybe_works, now=probe_t)            detected_at = probe_t            break        except:            pass    print(f"{cd:>10}  {f't={detected_at}' if detected_at else 'not within 100s':>24}")print()print("Shorter cooldown detects recovery faster but wastes more probe calls.")print("Longer cooldown is more patient but delays recovery detection.")

## Section 6: Break ItThree failure scenarios, each targeting a different reliability pattern.

### Break It 1: The AI coding assistant works from an outdated printout`billing.py` gets edited at t=3600 (discount code support is added). The cache TTL is7200 seconds, so the pre-edit answer is still cached. The assistant confidently servesa wrong answer because nothing told the cache the file changed.**Hint 1**: What event should trigger cache invalidation?**Hint 2**: Log the age of every served answer so staleness is visible, not silent.**Production impact**: A customer asks "does billing support discount codes?" and getstold "no" -- 100 seconds after the feature shipped. The answer is wrong, but nothingin the response indicates it might be stale.**Interview follow-up**: "How would you detect that your AI assistant is serving stalecontext in production, without waiting for a user to complain?"#### Buggy version: no invalidation, no age logging

In [5]:
def ai_assistant_answer_v1(filestore, cache, filename, question, now):
    '''Buggy version: no invalidation on write, and no logging of how old the served
    content actually is -- staleness is silent.'''
    cached_content, _ = cache.get(filename, now)
    if cached_content is not None:
        content = cached_content
    else:
        content = filestore.get_file(filename, now)
        cache.set(filename, content, now)
    has_discount = "discount_code" in content
    verdict = "supports" if has_discount else "does NOT support"
    return f"{filename} {verdict} discount codes."


buggy_cache = TTLCache(ttl_seconds=7200)

print("--- Bug: file is edited, but the cache doesn't know ---\n")
print("t=0:    ", ai_assistant_answer_v1(filestore, buggy_cache, "billing.py", "does it support discounts?", now=0))
print("        (billing.py is edited at t=3600 to add discount support -- the cache is never told)")
print("t=3700: ", ai_assistant_answer_v1(filestore, buggy_cache, "billing.py", "does it support discounts?", now=3700))
print("\nThe file has supported discount codes for 100 seconds by t=3700, but the assistant")
print("still says it doesn't -- and nothing in the output above tells you the answer is stale.")


--- Bug: file is edited, but the cache doesn't know ---

t=0:     billing.py does NOT support discount codes.
        (billing.py is edited at t=3600 to add discount support -- the cache is never told)
t=3700:  billing.py does NOT support discount codes.

The file has supported discount codes for 100 seconds by t=3700, but the assistant
still says it doesn't -- and nothing in the output above tells you the answer is stale.


#### Fix: invalidate on write, log context age on every response

In [6]:
def ai_assistant_answer_v2(filestore, cache, filename, question, now, log):
    '''Fixed version: logs context age on EVERY response (so staleness is detectable even
    if it happens), and the codebase-edit event below invalidates the cache on write (so it
    mostly doesn't happen in the first place). Two-layer fix, not one.'''
    cached_content, cached_at = cache.get(filename, now)
    if cached_content is not None:
        content = cached_content
        context_age = now - cached_at
        source = "cache"
    else:
        content = filestore.get_file(filename, now)
        cache.set(filename, content, now)
        context_age = 0
        source = "live fetch"

    log.append({"filename": filename, "source": source, "context_age_seconds": context_age})

    has_discount = "discount_code" in content
    verdict = "supports" if has_discount else "does NOT support"
    return f"{filename} {verdict} discount codes. [context age: {context_age}s, source: {source}]"


def edit_file(cache, filename):
    '''What a real deploy/webhook would trigger: invalidate the cache entry the moment the
    underlying file changes, instead of waiting for TTL to catch up.'''
    cache.invalidate(filename)


fixed_cache = TTLCache(ttl_seconds=7200)
freshness_log = []

print("--- Fix: invalidate on write, log context age on every response ---\n")
print("t=0:    ", ai_assistant_answer_v2(filestore, fixed_cache, "billing.py", "does it support discounts?", now=0, log=freshness_log))
edit_file(fixed_cache, "billing.py")
print("        (billing.py is edited at t=3600 -- this time the cache is invalidated immediately)")
print("t=3700: ", ai_assistant_answer_v2(filestore, fixed_cache, "billing.py", "does it support discounts?", now=3700, log=freshness_log))

print("\nFreshness log (this is what you'd actually check in production):")
for entry in freshness_log:
    print(" ", entry)


--- Fix: invalidate on write, log context age on every response ---

t=0:     billing.py does NOT support discount codes. [context age: 0s, source: live fetch]
        (billing.py is edited at t=3600 -- this time the cache is invalidated immediately)
t=3700:  billing.py supports discount codes. [context age: 0s, source: live fetch]

Freshness log (this is what you'd actually check in production):
  {'filename': 'billing.py', 'source': 'live fetch', 'context_age_seconds': 0}
  {'filename': 'billing.py', 'source': 'live fetch', 'context_age_seconds': 0}


### Break It 2: A tool that fails intermittentlyA 70%-failure-rate dependency kills most single-shot calls. Without retry, one failureends the task. But retry without backoff hammers the dependency, and retry without apredicate wastes time on permanent errors.**Hint 1**: Back off exponentially, not linearly. Linear growth stops helping when itmatters most.**Hint 2**: Check the HTTP status code before deciding to retry. A 401 is not worthfive attempts.**Production impact**: An agent retries a 401 (bad API key) five times with exponentialbackoff, adding 31 seconds of latency before the user sees an error that the firstresponse already contained.**Interview follow-up**: "Your agent retries a 401 error 5 times. Why is that worsethan failing immediately?"#### Buggy version: no retry

In [7]:
def make_flaky_tool(fail_probability: float, seed: int):
    rng = random.Random(seed)

    def flaky_tool():
        if rng.random() < fail_probability:
            raise ConnectionError("simulated transient failure")
        return "success"

    return flaky_tool


print("--- Bug: no retry, one failure ends the task ---\n")
flaky = make_flaky_tool(fail_probability=0.7, seed=1)
try:
    result = flaky()
    print("Result:", result)
except Exception as exc:
    print(f"Failed on the first attempt: {exc}")
    print("A 70%-fail-rate dependency will kill most single-shot calls to it -- unacceptable")
    print("for anything an agent needs to complete reliably.")


--- Bug: no retry, one failure ends the task ---

Failed on the first attempt: simulated transient failure
A 70%-fail-rate dependency will kill most single-shot calls to it -- unacceptable
for anything an agent needs to complete reliably.


### Break It 3: A cascading failureThe dependency is not flaky; it is fully down. Retrying a dead dependency does not helpit recover. It adds load to an already-failing system and wastes time on the caller'sside, potentially cascading the failure upstream.**Hint 1**: After how many consecutive failures should you stop trying?**Hint 2**: The half-open state is the part everyone forgets. Without it, the breakernever re-tests anything.**Production impact**: An agent makes 10 calls to a dead service, each timing out after30 seconds. That is 5 minutes of latency for zero benefit, and the user's request isblocked the entire time.**Interview follow-up**: "Your dependency is down. You have retry with backoff and acircuit breaker. Which do you use, and why not both?"#### Buggy version: keeps hammering a dead dependency

In [12]:
def make_always_failing_tool():
    def broken_tool():
        raise ConnectionError("dependency is down")
    return broken_tool


print("--- Bug: naive caller keeps hammering a fully-dead dependency ---\n")
broken = make_always_failing_tool()
attempted = 0
for t in range(10):
    try:
        broken()
    except Exception:
        pass
    attempted += 1
print(f"Made {attempted} calls to a dependency that never once succeeded.")
print("Every one of those wastes time on both sides -- a real network call here would add")
print("real latency to each failure, and do it 10 times over for zero benefit.")


--- Bug: naive caller keeps hammering a fully-dead dependency ---

Made 10 calls to a dependency that never once succeeded.
Every one of those wastes time on both sides -- a real network call here would add
real latency to each failure, and do it 10 times over for zero benefit.


#### Fix: circuit breaker fails fast

### Real rate-limit handlingReusing `agentlib.llm_client` from Chapter 1: with a real key present, fire a burst ofconcurrent calls large enough to trigger a real 429/overloaded response, then apply thesame `retry_with_backoff()` built above against the real API. Falls back to a simulatedflaky-tool exercise if no key is present.

In [16]:
if llm_client.HAS_KEY:
    import concurrent.futures

    def burst_call(i):
        return llm_client.call_model(
            messages=[{"role": "user", "content": f"Reply with only the number {i}."}],
            model=llm_client.DEFAULT_MODELS[llm_client.LLM_PROVIDER],
            max_tokens=10,
        )

    print("Firing 20 concurrent real API calls to try to trigger a real rate limit...\n")
    errors, successes = [], []
    with concurrent.futures.ThreadPoolExecutor(max_workers=20) as executor:
        futures = {executor.submit(burst_call, i): i for i in range(20)}
        for future in concurrent.futures.as_completed(futures):
            try:
                successes.append(future.result())
            except Exception as exc:
                errors.append(exc)

    print(f"{len(successes)} succeeded, {len(errors)} hit an error (rate limit or otherwise) on the first attempt.")
    if errors:
        print(f"Example real error: {errors[0]!r}")

    print("\nApplying retry_with_backoff() against the real API for one call:")
    robust_call = retry_with_backoff(lambda: burst_call(0), max_retries=5, sleep_fn=time.sleep)
    result, attempts_log = robust_call()
    for a in attempts_log:
        print(f"  attempt {a['attempt']} failed ({a['error']}) -- backed off {a['backoff_s']}s")
    print(f"Final result text: {result.text!r}")
else:
    print("No API key present -- reusing the simulated flaky-tool exercise above as the")
    print("CI/no-budget path instead of firing real concurrent calls.\n")
    simulated_flaky = make_flaky_tool(fail_probability=0.6, seed=7)
    robust_simulated = retry_with_backoff(simulated_flaky, max_retries=6)
    result, attempts_log = robust_simulated()
    for a in attempts_log:
        print(f"  attempt {a['attempt']} failed ({a['error']}) -- backed off {a['backoff_s']}s")
    print(f"Result: {result!r}, succeeded on attempt {len(attempts_log) + 1}")


No API key present -- reusing the simulated flaky-tool exercise above as the
CI/no-budget path instead of firing real concurrent calls.

  attempt 1 failed (simulated transient failure) -- backed off 1.07s
  attempt 2 failed (simulated transient failure) -- backed off 2.42s
Result: 'success', succeeded on attempt 3


## Section 7: Interview Q&A### Question 1: "How do you prevent cascading failures in a multi-service agent system?"**Model answer**: Circuit breakers around every inter-service call. Each breaker tracksconsecutive failures independently. When a downstream dependency fails past its threshold,the breaker opens and the caller fails fast instead of blocking on a dead service. Thehalf-open probe after cooldown detects recovery without a burst of retries. Combine withtimeouts on every outbound call so a slow dependency does not hold the caller indefinitely.### Question 2: "When should you retry an API call vs. fail immediately?"**Model answer**: Retry transient errors: 429 (rate limit), 408 (timeout), 5xx (servererror), and transport failures (ConnectionError, TimeoutError). Fail immediately onpermanent errors: 400 (bad request), 401 (bad credentials), 403 (forbidden), 404 (notfound). The distinction is whether a second attempt has any chance of succeeding. Retryinga 401 five times with exponential backoff adds 30 seconds of latency for zero benefit.### Question 3: "Your cache has a 5-minute TTL but the data changes every 30 seconds. What breaks?"**Model answer**: Every read in the 4.5-minute window between the last write and TTLexpiry serves stale data. The fix is not just shortening TTL (that increases load on thebacking store). Two-layer approach: (1) invalidate on write so the cache drops the entrythe moment the source changes, and (2) log context age on every response so stalenessis measurable even when invalidation misses an edge case. TTL becomes the safety net,not the primary freshness mechanism.### Question 4: "Explain the three states of a circuit breaker and when each transition happens."**Model answer**: Closed is normal operation; calls pass through and consecutive failuresare counted. When the count reaches the threshold, the breaker transitions to Open: allcalls fail immediately without reaching the dependency. After a cooldown period, thebreaker transitions to Half-Open and allows exactly one trial call. If that call succeeds,the breaker closes (normal operation resumes). If it fails, the breaker reopens with afresh cooldown. The half-open probe is the part that is easy to forget; without it thebreaker never detects recovery.### Debug-from-logs exercise"Users complain the AI coding assistant ignores recent code changes." Walk through howyou would debug context freshness in production, using the `freshness_log` this chaptergenerated above. What would you check first, and what in that log format tells you whetherthis is a caching bug (like break-it 1 above) vs. something else (e.g. the index has notre-ingested the file yet)?

### Reliability-pattern recall drillFor each scenario below, name the pattern (retry / circuit breaker / cache invalidation)that applies, and say what happens if you reach for the wrong one.

In [17]:
from agentlib.self_check import drill as open_drill

drill = open_drill(4)
drill.questions()

Chapter 4 written drill — 4 questions

1. A downstream service is fully down for the next 20 minutes during a deploy.
2. A network call fails about 1 in 20 times with no discernible pattern.
3. A document was updated five minutes ago, but an agent is still citing the old version.
4. A downstream service returns errors for 30 seconds during a brief traffic spike, then fully
   recovers on its own.


#### Answering theseWrite your answer into the slot for each question, run the cell, then use `drill.check(n)`to see your answer and the model answer side by side. `check(n)` will not show you ananswer until you have written one of your own. If you want it anyway, `drill.reveal(n)` isthere and makes no judgement.

In [18]:
# One slot per question. Replace the placeholder text, then run this cell.
# Anything under 25 words, or left as the placeholder, is not recorded.

# Question 1
drill.attempt(1, '''
(Your answer here.)
''')

# Question 2
drill.attempt(2, '''
(Your answer here.)
''')

# Question 3
drill.attempt(3, '''
(Your answer here.)
''')

# Question 4
drill.attempt(4, '''
(Your answer here.)
''')

print()
drill.status()

  1. not recorded — it is still the placeholder
  2. not recorded — it is still the placeholder
  3. not recorded — it is still the placeholder
  4. not recorded — it is still the placeholder

Chapter 4: 0/4 answered
  still open: [1, 2, 3, 4]


In [19]:
# Your answer, then the model answer. Change the number to work through the rest.
drill.check(1)

Question 1 has no recorded answer yet.

  A downstream service is fully down for the next 20 minutes during a deploy.

Write one with attempt() first. Reading the model answer before you have committed to your own turns this into a reading exercise -- once you have seen it you can no longer find out what you actually knew.
(If you really want it anyway: reveal(1).)


## Section 8: References1. Nygard, M. (2007). *Release It!* Pragmatic Bookshelf. Circuit breaker pattern   (Chapter 5), stability patterns for production systems.2. AWS Well-Architected Framework -- Reliability Pillar: retry with backoff, circuit   breaker, and graceful degradation patterns.   https://docs.aws.amazon.com/wellarchitected/latest/reliability-pillar/3. Stripe API docs -- Retry behavior and idempotency keys.   https://docs.stripe.com/error-handling#retries4. Google Cloud -- Retry strategy best practices.   https://cloud.google.com/storage/docs/retry-strategy5. Resilience4j (successor to Netflix Hystrix) -- CircuitBreaker, Retry, and   RateLimiter modules. https://resilience4j.readme.io/Related chapters:- Chapter 1 (tool calling, `agentlib.llm_client`)- Chapter 5 (cost of retries and cache misses)- Chapter 9 (deployment patterns, canary releases)

## Next: Chapter 5, Cost, Performance, and Model SelectionThis chapter was about surviving failures. Chapter 5 is about the cost of not failing:token economics, latency decomposition, and matching model choice to task complexity.